# Importing libraries

In [2]:
import sys
sys.path.insert(0, '..')
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from streamlit_app.model import PneumoniaCNN

## Loading the trained Model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# st.write(f"Using device: **{device}**")

In [6]:
def load_model():
    model = PneumoniaCNN().to(device)
    model.load_state_dict(torch.load(r"D:\Project\AI That Explains Medical Images Like a Doctor\Another way\model creation\pneumonia_cnn.pth", map_location=device))
    model.eval()
    return model

model = load_model()

C:\Users\deven\AppData\Local\Temp\ipykernel_4204\1627220880.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"D:\Project\AI That Explain

## Trasform

In [7]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])


## Defing the prediction function

In [8]:
def predict_xray(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = F.softmax(outputs, dim=1)

    confidence, predicted_class = torch.max(probs, 1)

    label = "Pneumonia" if predicted_class.item() == 1 else "Normal"

    return {
        "prediction": label,
        "confidence": float(confidence.item())
    }

## Medical summary generator

In [9]:
def generate_medical_summary(result):
    prediction = result["prediction"]
    confidence = result["confidence"] * 100

    if prediction == "Pneumonia":
        summary = f"""
        Diagnosis: Pneumonia
        Confidence: {confidence:.2f}%
        Lung Opacity: Detected
        Affected Area: Possible lower lung region
        Severity: Moderate
        """
    else:
        summary = f"""
        Diagnosis: Normal
        Confidence: {confidence:.2f}%
        Lung Opacity: Not detected
        """

    return summary.strip()


## LLM Explanation Generator

In [10]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")

def generate_explanation(medical_summary):
    prompt = f"""
    You are a professional radiology assistant.
    Explain the following chest X-ray findings in simple,
    doctor-like language that a patient can understand.
    
    Avoid giving absolute diagnosis.
    Add a medical disclaimer.

    Findings:
    {medical_summary}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content


ModuleNotFoundError: No module named 'openai'